# Розділ 7: Створення чат-застосунків
## Швидкий старт з OpenAI API

Ця записна книга адаптована з [репозиторію прикладів Azure OpenAI](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst), який містить записні книги для доступу до сервісів [Azure OpenAI](notebook-azure-openai.ipynb).

Python OpenAI API також працює з моделями Azure OpenAI, з деякими модифікаціями. Дізнайтесь більше про відмінності тут: [Як переключатися між кінцевими точками OpenAI та Azure OpenAI з Python](https://learn.microsoft.com/azure/ai-services/openai/how-to/switching-endpoints?WT.mc_id=academic-109527-jasmineg)


# Огляд  
"Великі мовні моделі — це функції, які відображають текст у текст. Маючи вхідний рядок тексту, велика мовна модель намагається передбачити текст, який з’явиться наступним"(1). Цей блокнот "швидкого старту" познайомить користувачів із загальними поняттями LLM, основними вимогами пакету для початку роботи з AML, м’яким введенням у дизайн підказок і кількома короткими прикладами різних випадків використання. 


## Зміст  

[Огляд](#overview)  
[Як користуватися сервісом OpenAI](#how-to-use-openai-service)  
[1. Створення вашого сервісу OpenAI](#1.-creating-your-openai-service)  
[2. Встановлення](#2.-installation)    
[3. Облікові дані](#3.-credentials)  

[Випадки використання](#use-cases)    
[1. Підсумовування тексту](#1.-summarize-text)  
[2. Класифікація тексту](#2.-classify-text)  
[3. Генерація нових назв продуктів](#3.-generate-new-product-names)  
[4. Тонке налаштування класифікатора](#4.fine-tune-a-classifier)  

[Посилання](#references)


### Створіть свій перший промпт  
Ця коротка вправа надасть базове введення для надсилання промптів у модель OpenAI для простої задачі "підсумовування".


**Кроки**:  
1. Встановіть бібліотеку OpenAI у ваше середовище Python  
2. Завантажте стандартні допоміжні бібліотеки та встановіть ваші типові облікові дані безпеки OpenAI для створеного сервісу OpenAI  
3. Виберіть модель для вашого завдання  
4. Створіть простий промпт для моделі  
5. Надішліть ваш запит до API моделі!


### 1. Встановлення OpenAI


In [ ]:
%pip install openai python-dotenv

### 2. Імпортуйте допоміжні бібліотеки та створіть облікові дані


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("OPENAI_API_KEY","")
assert API_KEY, "ERROR: OpenAI Key is missing"

client = OpenAI(
    api_key=API_KEY
    )


### 3. Пошук відповідної моделі  
Моделі на кшталт GPT-4o та GPT-4o mini можуть розуміти та генерувати природню мову, і доступні на платформі OpenAI з різними рівнями потужності та швидкості, що підходять для різних завдань.


In [ ]:
# Select a general purpose chat model
model = "gpt-4o-mini"


## 4. Проєктування запитів  

"Магія великих мовних моделей полягає в тому, що, навчаючись мінімізувати цю помилку прогнозування на великих обсягах тексту, моделі зрештою вивчають корисні для цих прогнозів концепції. Наприклад, вони вивчають концепції, як-от"(1):

* як правильно писати
* як працює граматика
* як перефразовувати
* як відповідати на питання
* як вести розмову
* як писати багатьма мовами
* як програмувати
* тощо.

#### Як контролювати велику мовну модель  
"З усіх входів до великої мовної моделі найбільший вплив чинить текстовий запит(1).

Великі мовні моделі можна підштовхувати до створення результату кількома способами:

Інструкція: Скажіть моделі, що ви хочете
Завершення: Спонукайте модель доповнити початок того, що ви хочете
Демонстрація: Покажіть моделі, що ви хочете, за допомогою:
Кількох прикладів у запиті
Багатьох сотень чи тисяч прикладів у датасеті для доточного налаштування"



#### Існують три основні рекомендації для створення запитів:

**Пояснюйте і показуйте**. Зробіть ясно, чого ви хочете, через інструкції, приклади або їх комбінацію. Якщо ви хочете, щоб модель відсортувала список за алфавітом або класифікувала абзац за настроєм, покажіть їй, що це саме те, що ви хочете.

**Надавайте якісні дані**. Якщо ви намагаєтеся створити класифікатор або змусити модель слідувати певному патерну, переконайтеся, що є достатньо прикладів. Обов’язково перевіряйте свої приклади — модель зазвичай досить розумна, щоб побачити базові орфографічні помилки та дати вам відповідь, але вона також може вважати, що це зроблено навмисно, і це вплине на відповідь.

**Перевіряйте налаштування.** Параметри temperature і top_p керують тим, наскільки детермінованою є модель у створенні відповіді. Якщо ви просите відповідь, де є лише одна правильна відповідь, краще встановити їх на нижчі значення. Якщо ви шукаєте більш різноманітні відповіді, тоді варто встановити їх вище. Найпоширеніша помилка – вважати ці налаштування регуляторами «кмітливості» чи «креативності».


Джерело: https://learn.microsoft.com/azure/ai-services/openai/overview


### 5. Надіслати!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


### Повторіть той самий виклик, як порівнюються результати?


In [ ]:

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


## Підсумуй текст  
#### Виклик  
Підсумуйте текст, додавши "tl;dr:" наприкінці уривка тексту. Зверніть увагу, як модель розуміє, як виконувати низку завдань без додаткових інструкцій. Ви можете експериментувати з більш описовими підказками, ніж tl;dr, щоб змінити поведінку моделі та налаштувати отримане підсумування(3).  

Останні дослідження показали значні покращення у багатьох NLP-завданнях та тестах шляхом попереднього тренування на великому корпусі тексту, а потім донавчання на конкретному завданні. Хоча зазвичай архітектура є загальною для завдань, цей метод все одно вимагає наборів даних для донавчання, специфічних для завдання, що містять тисячі або десятки тисяч прикладів. Натомість люди здатні виконувати нове мовне завдання, маючи лише кілька прикладів або прості інструкції — чого сучасні системи NLP здебільшого ще не можуть. Тут ми показуємо, що масштабування мовних моделей значно покращує загальнозавданісне виконання з небагатьма прикладами, іноді навіть досягаючи конкурентоспроможності з попередніми передовими підходами до донавчання. 



Підсумок  


# Вправи для кількох випадків використання  
1. Підсумувати текст  
2. Класифікувати текст  
3. Створити нові назви продуктів


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## Класифікація тексту  
#### Виклик  
Класифікуйте елементи за категоріями, наданими під час виконання. У наступному прикладі ми надаємо і категорії, і текст для класифікації в запиті (*playground_reference). 

Запит клієнта: Вітаю, одна з клавіш на клавіатурі мого ноутбука нещодавно зламалась, і мені потрібна заміна:

Класифікована категорія:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## Генерація нових назв продуктів
#### Виклик
Створіть назви продуктів на основі прикладів слів. Тут ми включаємо у підказку інформацію про продукт, для якого збираємося генерувати назви. Ми також надаємо подібний приклад, щоб показати шаблон, який хочемо отримати. Ми також встановили високе значення температури, щоб збільшити випадковість і отримати більш інноваційні відповіді.

Опис продукту: Домашній прилад для приготування молочних коктейлів
Початкові слова: швидкий, здоровий, компактний.
Назви продуктів: HomeShaker, Fit Shaker, QuickShake, Shake Maker

Опис продукту: Пара взуття, що підходить для будь-якого розміру стопи.
Початкові слова: адаптивний, підходить, омні-підходить.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}],
  store=False,)

response.output_text


# Посилання  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Портал Microsoft Foundry](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [Кращі практики для тонкого налаштування моделей GPT для класифікації тексту](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# Для додаткової допомоги  
[OpenAI Commercialization Team](AzureOpenAITeam@microsoft.com) 


# Учасники
* Louis Li  


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Відмова від відповідальності**:
Цей документ було перекладено за допомогою сервісу штучного інтелекту для перекладу [Co-op Translator](https://github.com/Azure/co-op-translator). Хоча ми прагнемо до точності, будь ласка, майте на увазі, що автоматичні переклади можуть містити помилки або неточності. Оригінальний документ рідною мовою слід вважати авторитетним джерелом. Для критично важливої інформації рекомендується професійний людський переклад. Ми не несемо відповідальності за будь-які непорозуміння або неправильні тлумачення, що виникли внаслідок використання цього перекладу.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
